# Sprint 1 — Data Pipeline
### AI-Powered YouTube Title Optimizer — Dataset Acquisition, Cleaning, Feature Engineering, Target Variable
Reference: `SOURCE_OF_TRUTH.md` §15 | `PLAN_IMPLEMENTATION.md` Sprint 1

Run top to bottom in Colab. Outputs `labeled_titles.csv` at the end — download it and feed it into `model_training.ipynb`.

## 0. Setup

In [ ]:
!pip -q install kaggle emoji nltk spacy pandas pyarrow matplotlib seaborn
!python -m spacy download en_core_web_sm -q

In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", IN_COLAB)

### Kaggle authentication
Uses Colab Secrets if you've stored `KAGGLE_USERNAME` and `KAGGLE_KEY` there (key icon in the left sidebar — remember to toggle "Notebook access" on for each). Falls back to a `kaggle.json` upload prompt if secrets aren't found.

In [ ]:
kaggle_authenticated = False

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        kaggle_authenticated = True
        print("Authenticated via Colab Secrets (KAGGLE_USERNAME / KAGGLE_KEY).")
    except Exception as e:
        print(f"Colab Secrets not set or not accessible ({e}); falling back to kaggle.json upload.")

if IN_COLAB and not kaggle_authenticated:
    from google.colab import files
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    if not (kaggle_dir / "kaggle.json").exists():
        uploaded = files.upload()  # select kaggle.json
        for fname in uploaded:
            (kaggle_dir / "kaggle.json").write_bytes(uploaded[fname])
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    kaggle_authenticated = True

if not IN_COLAB:
    print("Not in Colab — assuming kaggle.json is already configured locally (~/.kaggle/kaggle.json).")

## 1. Data Acquisition (SOT §15.1, PLAN §1.1)

- Source 1: `rsrishav/youtube-trending-video-dataset` — US slice only (full dataset is multi-GB across countries)
- Source 2: `datasnaek/youtube-new` — small, historical US/CA/GB/IN
- Source 3 (YouTube Data API scrape of MrBeast/Veritasium/Mark Rober/Kurzgesagt) and Source 5 (Webis Clickbait Corpus) are optional add-ons — see the last two cells in this section.

In [ ]:
import kaggle
import zipfile

kaggle.api.authenticate()

rs_target = RAW_DIR / "youtube-trending-video-dataset"
rs_target.mkdir(exist_ok=True)
for fname in ["US_youtube_trending_data.csv", "US_category_id.json"]:
    print(f"Downloading {fname}...")
    kaggle.api.dataset_download_file(
        "rsrishav/youtube-trending-video-dataset", fname, path=str(rs_target)
    )
zip_path = rs_target / "US_youtube_trending_data.csv.zip"
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(rs_target)
    zip_path.unlink()

ds_target = RAW_DIR / "youtube-new"
ds_target.mkdir(exist_ok=True)
print("Downloading datasnaek/youtube-new...")
kaggle.api.dataset_download_files("datasnaek/youtube-new", path=str(ds_target), unzip=True, quiet=False)

print("Done.")

### (Optional) Source 3 — YouTube Data API channel scrape
**Not required for Sprint 1 (pCTR model training)** — Sources 1+2 (Kaggle) are sufficient on their own. This is only needed later, in Sprint 3, to build the few-shot Prompt Database from high-performing channel titles (MrBeast, Veritasium, Mark Rober, Kurzgesagt). Safe to skip for now; the cell below auto-skips if no key is found (checks Colab Secret `YOUTUBE_API_KEY` first, then the environment variable).

In [ ]:
YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY", "")
if IN_COLAB and not YOUTUBE_API_KEY:
    try:
        from google.colab import userdata
        YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
    except Exception:
        pass

CHANNELS = {
    "MrBeast": "UCX6OQ3DkcsbYNE6H8uQQuVA",
    "Veritasium": "UCHnyfMqiRRG1u-2MsSQLbXA",
    "Mark Rober": "UCY1kMZp36IQSyNx_9h4mpCg",
    "Kurzgesagt": "UCsXVk37bltHxD1rDPwtNM8Q",
}

if YOUTUBE_API_KEY:
    !pip -q install google-api-python-client
    from googleapiclient.discovery import build
    import pandas as pd

    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    rows = []
    for name, channel_id in CHANNELS.items():
        uploads_playlist = (
            youtube.channels().list(part="contentDetails", id=channel_id).execute()
            ["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
        )
        video_ids, next_page = [], None
        while len(video_ids) < 500:
            resp = youtube.playlistItems().list(
                part="contentDetails", playlistId=uploads_playlist,
                maxResults=50, pageToken=next_page,
            ).execute()
            video_ids += [i["contentDetails"]["videoId"] for i in resp["items"]]
            next_page = resp.get("nextPageToken")
            if not next_page:
                break
        for i in range(0, len(video_ids[:500]), 50):
            batch = video_ids[i:i + 50]
            stats = youtube.videos().list(part="snippet,statistics", id=",".join(batch)).execute()
            for item in stats["items"]:
                rows.append({
                    "channel": name, "video_id": item["id"], "title": item["snippet"]["title"],
                    "views": int(item["statistics"].get("viewCount", 0)),
                    "likes": int(item["statistics"].get("likeCount", 0)),
                })
    pd.DataFrame(rows).to_csv(RAW_DIR / "channel_scrape.csv", index=False)
    print(f"Saved {len(rows)} rows -> {RAW_DIR / 'channel_scrape.csv'}")
else:
    print("YOUTUBE_API_KEY not set — skipping channel scrape (not needed until Sprint 3).")

### (Optional) Source 5 — Webis Clickbait Corpus
Requires manual registration at https://webis.de/data/webis-clickbait-17.html — download and place the archive under `data/raw/webis_clickbait/`, then unzip. Used later (Sprint 3) as negative few-shot examples, not required for pCTR model training.

## 2. Data Cleaning (SOT §15.4, PLAN §1.2)
- De-duplicate trending entries (collapse per-day rows to peak metrics)
- Remove videos with < 1,000 views
- Remove non-English titles
- Remove pure SEO keyword strings
- Preserve emojis (handled at feature stage, not stripped here)
- Flag (not drop) toxic titles for exclusion from the future few-shot DB

In [ ]:
import pandas as pd
import re

MIN_VIEWS = 1000

def load_rsrishav():
    path = RAW_DIR / "youtube-trending-video-dataset" / "US_youtube_trending_data.csv"
    df = pd.read_csv(path)
    df = df.rename(columns={
        "channelTitle": "channel_title",
        "categoryId": "category_id",
        "view_count": "views",
    })
    return df[["video_id", "title", "channel_title", "category_id", "views", "likes",
               "comment_count", "trending_date", "description"]]

def load_datasnaek():
    files = list((RAW_DIR / "youtube-new").glob("*videos.csv"))
    frames = []
    for f in files:
        try:
            d = pd.read_csv(f, encoding="latin1")
        except Exception:
            d = pd.read_csv(f, encoding="utf-8", encoding_errors="ignore")
        frames.append(d[["video_id", "title", "channel_title", "category_id", "views", "likes",
                          "comment_count", "trending_date", "description"]])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def dedupe_trending(df):
    return df.groupby("video_id", as_index=False).agg({
        "title": "last", "channel_title": "last", "category_id": "last",
        "views": "max", "likes": "max", "comment_count": "max", "description": "last",
    })

def is_seo_string(title):
    words = re.findall(r"[A-Za-z]+", str(title))
    if len(words) < 2:
        return True
    ratio = (title.count("|") + title.count("-")) / max(len(title), 1)
    return ratio > 0.15

def english_ascii_ratio(series: pd.Series, min_ratio: float = 0.85) -> pd.Series:
    """Vectorized English proxy: fraction of ASCII characters in the title.

    Both source datasets are already US-region trending videos, so the vast
    majority of titles are English. Row-by-row `langdetect` calls (~1-2ms
    each, no batching) take 10-20+ minutes at this scale; an ASCII-ratio
    heuristic gives an equivalent practical filter in a couple of seconds.
    Documented as a limitation (SOT §20) — a small number of English titles
    with heavy emoji/accents may be dropped, and vice versa.
    """
    ascii_count = series.str.encode("ascii", errors="ignore").str.decode("ascii").str.len()
    return (ascii_count / series.str.len().clip(lower=1)) >= min_ratio

def clean(df):
    df = df.dropna(subset=["title"]).copy()
    df = df[df["views"] >= MIN_VIEWS]
    df = df[~df["title"].apply(is_seo_string)]
    df = df[english_ascii_ratio(df["title"])]
    df = df.drop_duplicates(subset=["video_id"])
    return df.reset_index(drop=True)

frames = []
rs_path = RAW_DIR / "youtube-trending-video-dataset" / "US_youtube_trending_data.csv"
if rs_path.exists():
    frames.append(load_rsrishav())
ds = load_datasnaek()
if not ds.empty:
    frames.append(ds)

combined = pd.concat(frames, ignore_index=True)
print(f"Combined raw rows: {len(combined)}")
deduped = dedupe_trending(combined)
print(f"After de-dup: {len(deduped)}")
cleaned = clean(deduped)
print(f"After clean (views/SEO/lang): {len(cleaned)}")

In [ ]:
# Toxicity filter (unitary/toxic-bert) — flags rows for exclusion from the future few-shot DB (Sprint 3).
!pip -q install "transformers>=4.40" "accelerate>=0.30" "huggingface_hub>=0.23"

import torch
from huggingface_hub import login
from tqdm.auto import tqdm

if IN_COLAB:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_KEY"))
        print("Authenticated with Hugging Face Hub via Colab Secret HF_KEY.")
    except Exception as e:
        print(f"HF_KEY secret not set ({e}) — downloading unauthenticated (slower, rate-limited).")

use_gpu = torch.cuda.is_available()
print(f"GPU available: {use_gpu}" + ("" if use_gpu else
      " — Runtime > Change runtime type > T4 GPU will make this step much faster."))

try:
    from transformers import pipeline

    device = 0 if use_gpu else -1
    clf = pipeline("text-classification", model="unitary/toxic-bert", top_k=None, device=device)

    titles = cleaned["title"].tolist()
    batch_size = 64
    toxicity_scores = []
    for i in tqdm(range(0, len(titles), batch_size)):
        batch = titles[i:i + batch_size]
        out = clf(batch, batch_size=batch_size, truncation=True)
        toxicity_scores.extend(max(s["score"] for s in row if s["label"] == "toxic") for row in out)
    cleaned["toxicity_score"] = toxicity_scores
except Exception as e:
    print(f"Toxicity filter unavailable ({e}); defaulting toxicity_score=0.0")
    cleaned["toxicity_score"] = 0.0

cleaned["is_toxic"] = cleaned["toxicity_score"] >= 0.7
print(f"Toxic rows flagged: {int(cleaned['is_toxic'].sum())} / {len(cleaned)}")

## 2.5 Exploratory Data Analysis — Raw / Cleaned Data
Sanity-check the cleaned dataset before spending compute on feature engineering: missingness, view distribution, category balance, channel concentration, and the views/likes relationship.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Shape:", cleaned.shape)
print("\nMissing values (non-zero only):")
missing = cleaned.isna().sum()
print(missing[missing > 0] if missing.sum() else "None")

print("\nSummary stats — views/likes/comment_count:")
cleaned[["views", "likes", "comment_count"]].describe()

In [ ]:
# Views distribution (log scale — view counts are heavily right-skewed)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(cleaned["views"], bins=60, log_scale=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Views Distribution (log scale)")
axes[0].set_xlabel("Views")

sns.histplot(cleaned["likes"] + 1, bins=60, log_scale=True, ax=axes[1], color="#DD8452")
axes[1].set_title("Likes Distribution (log scale, +1 smoothing)")
axes[1].set_xlabel("Likes")

plt.tight_layout()
plt.show()

In [ ]:
# Category distribution — baseline CTR varies by category (SOT §15.2)
top_categories = cleaned["category_id"].value_counts().head(15)

plt.figure(figsize=(9, 5))
sns.barplot(x=top_categories.values, y=top_categories.index.astype(str), color="#55A868")
plt.title("Top 15 Categories by Row Count")
plt.xlabel("Count")
plt.ylabel("category_id")
plt.tight_layout()
plt.show()

In [ ]:
# Channel concentration — how much of the cleaned dataset comes from a small number of channels
top_channels = cleaned["channel_title"].value_counts().head(15)

plt.figure(figsize=(9, 5))
sns.barplot(x=top_channels.values, y=top_channels.index, color="#C44E52")
plt.title("Top 15 Channels by Row Count")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

print(f"\nUnique channels: {cleaned['channel_title'].nunique()}")
print(f"Top 15 channels account for {top_channels.sum() / len(cleaned):.1%} of rows")

In [ ]:
# Views vs likes — expect a strong positive log-log relationship; outliers are worth a manual look
plt.figure(figsize=(6.5, 5.5))
sns.scatterplot(data=cleaned.sample(min(5000, len(cleaned)), random_state=42),
                 x="views", y="likes", alpha=0.3, s=15, color="#8172B2")
plt.xscale("log")
plt.yscale("log")
plt.title("Views vs Likes (log-log, 5K sample)")
plt.tight_layout()
plt.show()

print("Longest and shortest titles:")
print(cleaned.loc[cleaned["title"].str.len().idxmax(), "title"])
print(cleaned.loc[cleaned["title"].str.len().idxmin(), "title"])

## 3. Feature Engineering (SOT §15.2, PLAN §1.3)

In [ ]:
import emoji
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import spacy

nltk.download("vader_lexicon", quiet=True)
vader = SentimentIntensityAnalyzer()
nlp = spacy.load("en_core_web_sm")

NUMBER_RE = re.compile(r"\d+")
SUB_TIER_BINS = [0, 10_000, 100_000, 1_000_000, float("inf")]
SUB_TIER_LABELS = ["0-10K", "10K-100K", "100K-1M", "1M+"]

def number_value(title):
    m = NUMBER_RE.search(title.replace(",", ""))
    return float(m.group()) if m else 0.0

def first_word_type(title):
    words = title.split()
    if not words:
        return "UNKNOWN"
    doc = nlp(words[0])
    return doc[0].pos_ if len(doc) else "UNKNOWN"

def all_caps_ratio(title):
    words = [w for w in re.findall(r"[A-Za-z]+", title) if len(w) > 1]
    if not words:
        return 0.0
    return sum(1 for w in words if w.isupper()) / len(words)

def build_features(df, title_col="title", subscriber_col="subscriber_count"):
    out = df.copy()
    out["title_length_chars"] = out[title_col].str.len()
    out["title_word_count"] = out[title_col].str.split().apply(len)
    out["has_number"] = out[title_col].apply(lambda t: bool(NUMBER_RE.search(t)))
    out["number_value"] = out[title_col].apply(number_value)

    vs = out[title_col].apply(lambda t: vader.polarity_scores(t))
    out["vader_compound"] = vs.apply(lambda d: d["compound"])
    out["vader_positive"] = vs.apply(lambda d: d["pos"])

    out["first_word_type"] = out[title_col].apply(first_word_type)
    out["has_question"] = out[title_col].str.contains("\\?")
    out["has_exclamation"] = out[title_col].str.contains("!")
    out["all_caps_ratio"] = out[title_col].apply(all_caps_ratio)
    out["has_emoji"] = out[title_col].apply(lambda t: any(c in emoji.EMOJI_DATA for c in t))

    if subscriber_col in out.columns:
        out["channel_subscriber_tier"] = pd.cut(
            out[subscriber_col], bins=SUB_TIER_BINS, labels=SUB_TIER_LABELS, right=False
        ).astype(str)
    else:
        out["channel_subscriber_tier"] = SUB_TIER_LABELS[0]
    return out

featured = build_features(cleaned)
print(f"Featured: {len(featured)} rows, {len(featured.columns)} cols")
featured.head()

## 4. Target Variable — Virality Score (SOT §15.3, PLAN §1.4)
`Virality_Score = Views / Subscriber_Count`. The Kaggle sources don't ship subscriber counts, so we fall back to a category-normalized proxy (`views / median views for that category_id`) — documented as a limitation (SOT §20).

In [ ]:
import numpy as np

def compute_virality_score(df):
    out = df.copy()
    if "subscriber_count" in out.columns and out["subscriber_count"].notna().any():
        out["virality_score"] = out["views"] / out["subscriber_count"].replace(0, np.nan)
    else:
        cat_median = out.groupby("category_id")["views"].transform("median")
        out["virality_score"] = out["views"] / cat_median.replace(0, np.nan)
    out = out.dropna(subset=["virality_score"])
    return out[np.isfinite(out["virality_score"])]

labeled = compute_virality_score(featured)
threshold = labeled["virality_score"].median()
labeled["label"] = (labeled["virality_score"] >= threshold).astype(int)

print(f"Median threshold: {threshold:.4f}")
print(labeled["label"].value_counts(normalize=True))

## 5. Exploratory Data Analysis — Engineered Features & Target
Validates the SOT §16.4 hypothesis (VADER, has_number, title length are top signals) before training, and checks the bell-curve title-length effect (Kawata et al., 2025) and class balance from the median split.

In [ ]:
# Virality_Score distribution + median threshold, and resulting class balance
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(labeled["virality_score"], bins=80, log_scale=True, ax=axes[0], color="#4C72B0")
axes[0].axvline(threshold, color="red", linestyle="--", label=f"median = {threshold:.3f}")
axes[0].set_title("Virality_Score Distribution (log scale)")
axes[0].legend()

sns.countplot(x=labeled["label"], ax=axes[1], hue=labeled["label"], palette="Set2", legend=False)
axes[1].set_title("Class Balance (0 = below median, 1 = above median)")
axes[1].set_xlabel("label")

plt.tight_layout()
plt.show()

In [ ]:
# Bell-curve check: title length/word count vs virality_score (Kawata et al., 2025 — SOT §2.3)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.regplot(data=labeled.sample(min(5000, len(labeled)), random_state=42),
            x="title_length_chars", y="virality_score", lowess=True,
            scatter_kws={"alpha": 0.2, "s": 12}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Title Length (chars) vs Virality_Score")

sns.regplot(data=labeled.sample(min(5000, len(labeled)), random_state=42),
            x="title_word_count", y="virality_score", lowess=True,
            scatter_kws={"alpha": 0.2, "s": 12}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Title Word Count vs Virality_Score")

plt.tight_layout()
plt.show()

In [ ]:
# VADER compound distribution — target is |compound| > 0.5 (Berger & Milkman, 2012 — SOT §1.5)
plt.figure(figsize=(7.5, 4.5))
sns.histplot(labeled["vader_compound"], bins=60, color="#4C72B0", kde=True)
plt.axvline(0.5, color="red", linestyle="--", label="high-arousal threshold (+0.5)")
plt.axvline(-0.5, color="red", linestyle="--")
plt.title("VADER Compound Score Distribution")
plt.legend()
plt.tight_layout()
plt.show()

pct_high_arousal = (labeled["vader_compound"].abs() > 0.5).mean()
print(f"Share of titles with |vader_compound| > 0.5: {pct_high_arousal:.1%}")

In [ ]:
# Boolean/categorical feature signals — does having a number/question/exclamation/emoji shift virality_score?
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
for ax, col in zip(axes, ["has_number", "has_question", "has_exclamation", "has_emoji"]):
    sns.boxplot(data=labeled, x=col, y="virality_score", hue=col, palette="Set2", legend=False, ax=ax)
    ax.set_yscale("log")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# first_word_type distribution — verb starts are hypothesized to outperform noun starts (SOT §15.2)
top_pos = labeled["first_word_type"].value_counts().head(10)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=top_pos.values, y=top_pos.index, color="#55A868")
plt.title("Top 10 First-Word POS Tags")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

print("\nMedian virality_score by first_word_type (top 10 by frequency):")
print(labeled[labeled["first_word_type"].isin(top_pos.index)]
      .groupby("first_word_type")["virality_score"].median().sort_values(ascending=False))

In [ ]:
# Correlation heatmap — numeric features vs. label (quick view of what the model will lean on)
numeric_cols = [
    "title_length_chars", "title_word_count", "has_number", "number_value",
    "vader_compound", "vader_positive", "has_question", "has_exclamation",
    "all_caps_ratio", "has_emoji", "virality_score", "label",
]
corr = labeled[numeric_cols].astype(float).corr()

plt.figure(figsize=(9, 7.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
out_path = PROCESSED_DIR / "labeled_titles.csv"
labeled.to_csv(out_path, index=False)
print(f"Saved -> {out_path} ({len(labeled)} rows)")

if IN_COLAB:
    from google.colab import files
    files.download(str(out_path))

**Next step:** open `model_training.ipynb`, upload `labeled_titles.csv`, and train the pCTR model.